# Thali — multi-task SmolVLA fine-tune (Kaggle, 2× T4 or P100)

Fine-tunes `lerobot/smolvla_base` on `Prashant-77/thali_all` (1050 scripted-expert episodes (742 837 frames), 7 skills, 3 cameras 240×320, 12-D actions, 10 paraphrases per skill) and pushes the checkpoint to `Prashant-77/thali_smolvla`.

**Before running:** add a Kaggle secret `HF_TOKEN` (write access to Prashant-77). Enable GPU (T4 ×2 or P100) and Internet in the notebook settings. Expected wall time ≈ 5–6 h for 20 000 steps at batch 16.

In [ ]:
# Pin the stack lerobot 0.4.4 was tested with locally (the Kaggle image ships transformers 5.x, which breaks it).
!pip -q install 'lerobot==0.4.4' 'transformers==4.57.6' 'huggingface_hub==0.35.3' 'accelerate==1.15.0' 'safetensors>=0.5' 'num2words'
import torch, lerobot, transformers, huggingface_hub
print(torch.__version__, torch.cuda.get_device_name(0), 'x', torch.cuda.device_count(), '| lerobot', lerobot.__version__, '| transformers', transformers.__version__, '| hub', huggingface_hub.__version__)
from transformers.utils.hub import is_offline_mode  # fails fast if the pins did not take

In [ ]:
import os, pathlib
# Hub credential: the notebook Secret when attached in the UI, else the private input dataset thali-secrets (kernels pushed via the API;
# its mount point differs between kernel types, so search /kaggle/input for the file)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    os.environ['HF_TOKEN'] = next(pathlib.Path('/kaggle/input').rglob('hf_token.txt')).read_text().strip()
from huggingface_hub import login, whoami
login(token=os.environ['HF_TOKEN']); print(whoami()['name'])

In [ ]:
# dataset: streamed from the Hub into the LeRobot cache
from lerobot.datasets.lerobot_dataset import LeRobotDataset
ds = LeRobotDataset('Prashant-77/thali_all')
print(ds.num_episodes, 'episodes', ds.num_frames, 'frames', ds.fps, 'fps'); print(ds.meta.features.keys()); print(ds.meta.tasks.head())

In [ ]:
# Training. SmolVLA config follows the plan (plan §3 Phase 3.1) with a batch that fits a T4 (16 GB): 16 x 3 cameras at 240x320
# (resized with padding to 512 by the policy), AMP on, vision encoder frozen (default), action expert + state projection trained.
# smolvla_base was pretrained with generic camera keys camera1/2/3; --rename_map maps our overhead / wrist_a / wrist_b onto them
# (the same map is applied at inference in runtime/executors.py).  subprocess + check=True stops the notebook on failure.
#
# Sessions: a T4 does ~8 s/step at batch 16 and Kaggle ends a session after 12 h, so each run does STEPS_PER_SESSION steps,
# starts from the checkpoint already on the Hub (Prashant-77/thali_smolvla) when there is one, saves every 500 steps, and the
# last cell pushes the newest checkpoint back.  Re-run the notebook until TRAINING_STEP.txt on the Hub reads the target.
import json, subprocess, sys
from huggingface_hub import HfApi
STEPS_PER_SESSION = 4500
RENAME = {"observation.images.overhead": "observation.images.camera1",
          "observation.images.wrist_a": "observation.images.camera2",
          "observation.images.wrist_b": "observation.images.camera3"}
files = [s.rfilename for s in HfApi().model_info('Prashant-77/thali_smolvla').siblings]
resume_from = 'Prashant-77/thali_smolvla' if 'model.safetensors' in files else 'lerobot/smolvla_base'
print('starting from', resume_from)
cmd = ['lerobot-train',
       f'--policy.path={resume_from}', '--policy.device=cuda', '--policy.use_amp=true',
       '--dataset.repo_id=Prashant-77/thali_all', f'--rename_map={json.dumps(RENAME)}',
       '--batch_size=16', f'--steps={STEPS_PER_SESSION}', '--log_freq=100', '--save_freq=500', '--eval_freq=0', '--num_workers=4',
       '--policy.chunk_size=50', '--policy.n_action_steps=50',
       '--output_dir=/kaggle/working/outputs/smolvla_mt', '--job_name=thali_smolvla', '--wandb.enable=false',
       '--policy.push_to_hub=false', '--seed=1000']
subprocess.run(cmd, check=True, stdout=sys.stdout, stderr=sys.stderr)

In [ ]:
# Push the newest checkpoint of this session to the Hub (the resume point for the next session) and a model card.
import glob, pathlib, requests
from huggingface_hub import HfApi
api = HfApi()
api.create_repo('Prashant-77/thali_smolvla', exist_ok=True)
ckpts = sorted(p for p in glob.glob('/kaggle/working/outputs/smolvla_mt/checkpoints/*/pretrained_model') if 'last' not in p)
src = ckpts[-1]
done_here = int(pathlib.Path(src).parent.name)
try:   # steps accumulated by earlier sessions
    prev = int(requests.get('https://huggingface.co/Prashant-77/thali_smolvla/resolve/main/TRAINING_STEP.txt', timeout=30).text.split()[1])
except Exception:
    prev = 0
total = prev + done_here
api.upload_folder(folder_path=src, repo_id='Prashant-77/thali_smolvla', commit_message=f'SmolVLA fine-tune: +{done_here} steps this session, {total} total (Kaggle T4, batch 16)')
api.upload_file(path_or_fileobj=f'step {total}\n'.encode(), path_in_repo='TRAINING_STEP.txt', repo_id='Prashant-77/thali_smolvla')
api.upload_file(path_or_fileobj=f'''---
license: apache-2.0
base_model: lerobot/smolvla_base
datasets: [Prashant-77/thali_all]
tags: [lerobot, smolvla, so101, bimanual, mujoco, thali]
---
# Thali multi-task SmolVLA
Fine-tuned on 1050 scripted-expert episodes of the Thali dinner-table task (7 skills, language-conditioned), batch 16, {total} steps so far
(trained in 12 h Kaggle T4 sessions of 4500 steps; camera keys renamed overhead/wrist_a/wrist_b -> camera1/2/3).
Evaluated back in the MuJoCo env by `eval/skill_eval.py --kind smolvla` and `eval/run_seeds.py --policy smolvla` in the Thali repo.
'''.encode(), path_in_repo='README.md', repo_id='Prashant-77/thali_smolvla')
print('pushed Prashant-77/thali_smolvla at total step', total)

## After it finishes
Back on the laptop: `make eval POLICY=smolvla` pulls `Prashant-77/thali_smolvla` and adds the SmolVLA rows to `results/seeds.json`. Until then those rows read "pending SmolVLA run".